# BrowserAgentTool — ComputerUseAgent inside the main Agent (v1.0.8)

The killer pattern: your **main `Agent` plans**, and when it needs to drive a browser it delegates to a sub-`ComputerUseAgent` via a single `browser_use` tool call.

- The main agent never has to think in pixels.
- The sub-agent never has to plan the larger task.
- Both are observable through the same event stream.
- The browser tool is just another `Tool` in your agent's toolbox — it composes with `WebSearchTool`, `PDFTool`, RAG, anything else.

This notebook walks through three real patterns:
1. Plain agent + `BrowserAgentTool` for one-off browser work
2. Agent with a curated toolset where `browser_use` is one option among many
3. `share_browser=True` — multi-step browser session reused across calls

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shipit_agent import Agent
from shipit_agent.computer_use import (
    BrowserAgentTool, MockBrowserSession,
)

## Example 1 — Plain Agent + browser tool

We build a `BrowserAgentTool` against a `MockBrowserSession` (so the notebook runs offline). In production, swap the factory for `PlaywrightBrowserSession.launch(...)`.

In [ ]:

class ScriptedLLM:
    def __init__(self, replies):
        self.replies = replies
        self._i = 0
    def complete(self, *, messages, **_):
        from shipit_agent.llms.base import LLMResponse
        text = self.replies[min(self._i, len(self.replies) - 1)]
        self._i += 1
        return LLMResponse(content=text)

class StringLLM:
    """For the sub-agent — accepts string complete() returns."""
    def __init__(self, replies):
        self.replies = replies
        self._i = 0
    def complete(self, *, messages, **_):
        text = self.replies[min(self._i, len(self.replies) - 1)]
        self._i += 1
        return text

In [ ]:
# The browser tool's sub-agent uses a vision-capable LLM (StringLLM here)
browser_tool = BrowserAgentTool(
    llm=StringLLM(replies=[
        'ACTION: navigate https://apple.com',
        'ACTION: click 240,80',
        'ACTION: done The iPhone 15 Pro starts at $999.',
    ]),
    browser_factory=lambda: MockBrowserSession(),
    max_iterations=8,
)

# Direct invocation of the tool — no main agent yet
result = browser_tool.run(goal='Find the iPhone 15 Pro starting price.')
print(result.text)
print()
print('metadata:')
for k in ('status', 'iterations'):
    print(f'  {k}: {result.metadata[k]}')

## Example 2 — Main Agent calls `browser_use` as one tool among many

The main agent decides when to use the browser vs simpler tools (web_search, pdf_extract, etc.). The `BrowserAgentTool` shows up alongside them — same Tool protocol.

In [ ]:
browser_tool_2 = BrowserAgentTool(
    llm=StringLLM(replies=[
        'ACTION: navigate https://example.com/contact',
        'ACTION: type "Inquiry from research bot"',
        'ACTION: done Form filled and submitted.',
    ]),
    browser_factory=lambda: MockBrowserSession(),
    max_iterations=6,
)

# Toolset visible to the main agent
agent = Agent(
    llm=ScriptedLLM(replies=['I will respond directly.']),
    tools=[browser_tool_2],
    auto_use_skills=False,
)

print('Tools the main agent sees:')
for t in agent._effective_tools('any'):
    print(f'  - {t.name}: {t.description[:80]}...')

print()
print('Tool schema (what the model sees):')
import json
print(json.dumps(agent._effective_tools('any')[0].schema(), indent=2))

## Example 3 — `share_browser=True` for multi-step browser sessions

When a workflow needs multiple browser interactions that share state (logged-in session, cookies, scroll position), enable `share_browser=True`. The same browser is reused across `tool.run()` calls until you `tool.close()` it.

Trade-off: faster (no relaunch) but state leaks between calls — only enable when the parent agent is aware that browser state persists.

In [ ]:
browsers_made = []
def factory():
    b = MockBrowserSession()
    browsers_made.append(b)
    return b

shared_tool = BrowserAgentTool(
    llm=StringLLM(replies=[
        'ACTION: done step 1 finished',
        'ACTION: done step 2 finished',
        'ACTION: done step 3 finished',
    ]),
    browser_factory=factory,
    share_browser=True,   # ← reuse the browser across calls
)

for goal in [
    'Log in to the dashboard',
    'Open the billing page',
    'Download the latest invoice',
]:
    out = shared_tool.run(goal=goal)
    print(f'  → {out.text.splitlines()[0]}')

print()
print(f'browsers created: {len(browsers_made)}  (would be 3 without share_browser)')

# Always close at the end
shared_tool.close()
print('shared browser closed')

## Production setup

The same code, but with Playwright + a real LLM:

```python
# pip install playwright
# playwright install chromium

from shipit_agent import Agent, VerifierNetwork
from shipit_agent.computer_use import (
    BrowserAgentTool, PlaywrightBrowserSession,
)

browser_tool = BrowserAgentTool(
    llm=opus_llm,                                  # vision-capable
    browser_factory=lambda: PlaywrightBrowserSession.launch(headless=True),
    max_iterations=12,
)

verifier = VerifierNetwork(llm=haiku_llm, goal='Research only — no purchases.')

agent = Agent(
    llm=opus_llm,                                  # main planner
    tools=[browser_tool, WebSearchTool(), PDFTool()],
    verifier=verifier,                             # gate destructive actions
)

result = agent.run(
    'Find the cheapest direct SFO-JFK flight on May 20 '
    'and summarise the booking page.'
)
```

## Why this is powerful

* **Composable** — `browser_use` is just a Tool. Add it to any Agent. Stack with RAG, structured output, verifier, memory consolidation.
* **Gated** — pair with `VerifierNetwork` and the verifier vetoes destructive browser actions before they run.
* **Observable** — every browser action lands in `result.metadata['actions']`, which flows through the parent agent's events.
* **Self-hosted** — no Devin/Operator subscription. Your LLM, your infra, your data.

→ See [Agent → ComputerUseAgent](https://docs.shipiit.com/agent/computer-use/) for the full reference.